# Interactive grouping для optbinning — демо

Віджет для перегляду і ручного редагування групувань (binning) після `optbinning`:

1. `optbinning.BinningProcess` робить базовий оптимальний групінг.
2. У віджеті вибираєте змінну → бачите WoE, кількості та binning table.
3. Знизу редагуєте **чернетку** (draft): об'єднуєте біни/групи, додаєте точки розбиття, переносите категорії, або просите optbinning переоптимізувати з обмеженнями.
4. **Apply** → змінна перераховується через `optbinning` (з фіксованими `user_splits`), графіки і таблиця оновлюються. Є Undo / Redo / Reset.
5. Результат: `w.to_binning_process()` → звичайний `BinningProcess` (наприклад для `Scorecard`), `w.transform(X)` → WoE, `w.save_json()`.

### 0. Середовище
Локально: `pip install -e .` у корені проєкту (або просто запустіть ноутбук з папки `notebooks/`).

**Google Colab**: завантажте папку `interactive_grouping/` (наприклад як `interactive_grouping.zip`) і розкоментуйте рядки нижче.

In [1]:
# --- Google Colab ---
# !pip -q install optbinning
# from google.colab import files; files.upload()      # interactive_grouping.zip
# !unzip -o -q interactive_grouping.zip

# Make the package importable without installation (notebook in notebooks/).
import os
import sys

for path in (".", ".."):
    package_dir = os.path.join(path, "interactive_grouping")
    if os.path.isdir(package_dir) and os.path.abspath(path) not in sys.path:
        sys.path.insert(0, os.path.abspath(path))

In [2]:
import warnings

from optbinning import BinningProcess

from interactive_grouping import InteractiveBinning, make_credit_data

warnings.filterwarnings("ignore")

### 1. Дані
Синтетичний кредитний датасет: числові й категоріальні змінні, пропуски (`NaN`) і спеціальний код `-1` у `income` («не вказано»). `default = 1` — подія.

In [3]:
X, y = make_credit_data(n=20_000, seed=42)
print(X.shape, "default rate:", round(y.mean(), 4))
X.head()

(20000, 9) default rate: 0.1561


,age,income,months_employed,utilization,late_payments,debt_to_income,home_ownership,purpose,region
0,47.0,51400.0,35.0,0.663,0.0,0.292,RENT,debt_consolidation,North
1,54.0,10600.0,16.0,0.433,0.0,0.412,MORTGAGE,debt_consolidation,West
2,45.0,134800.0,133.0,0.950,1.0,0.266,RENT,house,South
3,43.0,28100.0,14.0,0.651,2.0,0.432,RENT,education,Center
4,56.0,16000.0,99.0,0.749,0.0,0.604,MORTGAGE,medical,None


### 2. Базовий групінг optbinning

In [4]:
variables = list(X.columns)
categorical = ["home_ownership", "purpose", "region"]

bp = BinningProcess(variables, categorical_variables=categorical, special_codes=[-1])
bp.fit(X, y)
bp.summary()[["name", "dtype", "n_bins", "iv", "gini"]]

,name,dtype,n_bins,iv,gini
0,age,numerical,10,0.105434,0.140737
1,income,numerical,11,0.223586,0.256727
2,months_employed,numerical,7,0.046763,0.109955
3,utilization,numerical,9,0.202301,0.250224
4,late_payments,numerical,3,0.12033,0.147376
5,debt_to_income,numerical,7,0.045883,0.119444
6,home_ownership,categorical,3,0.082636,0.150191
7,purpose,categorical,7,0.02676,0.087544
8,region,categorical,5,0.000945,0.016958


### 3. Інтерактивний віджет (у стилі SAS Interactive Grouping)

* **Groupings**: ліворуч — дрібні (fine) біни, розфарбовані за групами (номер групи над смугою, червоні відрізки — WoE групи, тонка лінія — WoE дрібних бінів); праворуч — групи (кількість + WoE).
* *Detail Level = Coarse* — лише огляд груп, без змін.
* *Detail Level = Fine* — редагування. Виділіть дрібний бін (Ctrl/Shift+клік) і в **Assign selected to group** виберіть групу → **Assign**:
  * **числові**: попередня група (переходять усі дрібні біни від початку групи до виділеного), наступна група (виділений і всі наступні до кінця групи), нова група «this and higher» / «this and lower» (група ділиться в цьому місці — монотонність не ламається). Якщо група спорожніла — вона зливається з сусідньою;
  * **категоріальні**: будь-яка інша група або нова;
  * **Split selected fine bin** (числові) — ділить дрібний бін на два; групи не змінюються.
* **Variable Statistics**: Original (optbinning) / Applied / New. Нове групування застосовується лише після **Apply** (перерахунок через optbinning). **Variable Role = Rejected** виключає змінну з експорту.
* Вкладка **Variables** — IV/Gini усіх змінних (original vs new), збереження в JSON.
* `InteractiveBinning(..., advanced=True)` — додатковий блок (кількість дрібних бінів, ручний список меж, оптимізатор optbinning). **Експериментальний, не перевірений у реальній роботі — перевіряйте результат.**

Що спробувати: **age** — optbinning дав 10 груп з «шумним» горбом WoE; перенесіть дрібні біни між сусідніми групами або злийте групу з сусідньою. **purpose** — перенесіть категорію в іншу групу.

> Без інтернету (plotly.js з CDN) використовуйте `backend="matplotlib"`.

In [5]:
w = InteractiveBinning(bp, X, y)       # backend="matplotlib" as a fallback
w

### 4. Результати редагування

In [6]:
w.summary()

,name,role,dtype,n_bins,iv,strength,gini,trend,edited,original_n_bins,original_iv,original_gini
0,age,input,numerical,10,0.105434,medium,0.140737,valley,False,10,0.105434,0.140737
1,income,input,numerical,11,0.223586,medium,0.256727,descending,False,11,0.223586,0.256727
2,months_employed,input,numerical,7,0.046763,weak,0.109955,descending,False,7,0.046763,0.109955
3,utilization,input,numerical,9,0.202301,medium,0.250224,ascending,False,9,0.202301,0.250224
4,late_payments,input,numerical,3,0.120330,medium,0.147376,ascending,False,3,0.120330,0.147376
5,debt_to_income,input,numerical,7,0.045883,weak,0.119444,ascending,False,7,0.045883,0.119444
6,home_ownership,input,categorical,3,0.082636,weak,0.150191,-,False,3,0.082636,0.150191
7,purpose,input,categorical,7,0.026760,weak,0.087544,-,False,7,0.026760,0.087544
8,region,input,categorical,5,0.000945,useless,0.016958,-,False,5,0.000945,0.016958


In [7]:
# New optbinning BinningProcess with all manual groupings (fitted)
bp_edited = w.to_binning_process()
bp_edited.summary()[["name", "n_bins", "iv", "gini"]]

,name,n_bins,iv,gini
0,age,10,0.105434,0.140737
1,income,11,0.223586,0.256727
2,months_employed,7,0.046763,0.109955
3,utilization,9,0.202301,0.250224
4,late_payments,3,0.12033,0.147376
5,debt_to_income,7,0.045883,0.119444
6,home_ownership,3,0.082636,0.150191
7,purpose,7,0.02676,0.087544
8,region,5,0.000945,0.016958


In [8]:
# WoE transformation with the current groupings
X_woe = w.transform(X, metric="woe")
X_woe.head()

,age,income,months_employed,utilization,late_payments,debt_to_income,home_ownership,purpose,region
0,0.185815,0.465961,-0.091625,-0.284989,0.189415,0.102248,-0.281917,-0.010977,0.037054
1,0.152547,-0.918344,-0.162542,0.189012,0.189415,-0.096012,0.213861,-0.010977,-0.003037
2,0.185815,1.160538,0.249644,-0.821502,-0.225069,0.228128,-0.281917,0.274546,0.028999
3,0.185815,0.010500,-0.162542,-0.284989,-0.845327,-0.096012,-0.281917,-0.062300,-0.033994
4,0.106003,-0.556650,0.078645,-0.503805,0.189415,-0.451285,0.213861,-0.145590,0.000000


### 5. Далі — скоркарта на відредагованих групуваннях (стандартний optbinning)

In [9]:
from optbinning import Scorecard
from sklearn.linear_model import LogisticRegression

scorecard = Scorecard(binning_process=w.to_binning_process(fit=False),
                      estimator=LogisticRegression(max_iter=1000),
                      scaling_method="min_max", scaling_method_params={"min": 300, "max": 850})
scorecard.fit(X, y)
scorecard.table(style="summary").head(12)

,Variable,Bin,Points
0,age,"(-inf, 29.50)",52.068555
1,age,"[29.50, 32.50)",66.585825
2,age,"[32.50, 35.50)",70.464475
3,age,"[35.50, 38.50)",73.189361
4,age,"[38.50, 40.50)",83.589484
5,age,"[40.50, 50.50)",77.680385
6,age,"[50.50, 54.50)",75.640287
7,age,"[54.50, 58.50)",72.786014
8,age,"[58.50, 68.50)",54.565993
9,age,"[68.50, inf)",0.962275


### 6. Збереження / завантаження ручних групувань

In [10]:
w.save_json("binning_splits.json")

# later / in another session:
w2 = InteractiveBinning(BinningProcess(variables, categorical_variables=categorical,
                                       special_codes=[-1]), X, y)
print("problems:", w2.load_json("binning_splits.json"))
w2.summary()[["name", "iv", "edited"]]

problems: []


,name,iv,edited
0,age,0.105434,False
1,income,0.223586,False
2,months_employed,0.046763,False
3,utilization,0.202301,False
4,late_payments,0.120330,False
5,debt_to_income,0.045883,False
6,home_ownership,0.082636,False
7,purpose,0.026760,False
8,region,0.000945,False


### 7. Інші способи створення
```python
InteractiveBinning.from_data(X, y, categorical_variables=categorical, special_codes=[-1])
InteractiveBinning({"age": optb_age, "income": optb_income}, X, y)   # dict of OptimalBinning
w.binning_fit_params(only_edited=True)   # -> binning_fit_params for your own BinningProcess
```